In [ ]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

import torch

sys.path.append(".")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())

In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))

# SemEval-2016 Task 6: stance detection and opinion-target classification

Fine-tunes the pretrained checkpoints and the baselines on
[SemEval-2016 Task 6](https://alt.qcri.org/semeval2016/task6/), read from the local
`data-all-annotations/` files. Two tasks, over two of the three annotation columns:

- **stance** — `AGAINST` / `NONE` / `FAVOR`, does the tweeter support the target.
- **opinion** — `TARGET` / `OTHER` / `NO ONE`, who the tweet expresses an opinion about.

Both reuse the `bias_*` keys and `bias_head`, exactly as MITweet ideology does: three
exclusive classes over one 768→3 head. Stance is ordered with `NONE` in the middle so the
ordering matches the head's left/center/right, which is what makes an AllSides checkpoint's
head meaningful here without weight surgery.

Two knobs beyond the model:

- `prepend` is `target` (the target entity as sentence A, the tweet as sentence B, so only
  the tweet is ever truncated) or `none` as the control floor.
- `split` picks the test set: `taskA` (the same five targets as training) or `taskB`
  (Donald Trump, which appears in *no* training row — a cross-target evaluation).

Validation is the shipped `trialdata` throughout. It is small (100 rows) and covers only
Hillary Clinton and Legalization of Abortion, so it drives early stopping and nothing else;
critically it contains no Donald Trump rows, so the `taskB` target never reaches model
selection. `semeval._check_no_validation_leak` fails the run if that ever stops being true.

In [ ]:
import glob
from pathlib import Path

from huggingface_hub import snapshot_download

from config import load_run_config
from finetuning import aggregate as agg
from finetuning.experiments import run_semeval_experiment, ExperimentConfig
from finetuning.semeval import ALL_TARGETS, PREPEND_MODES, SemEvalConfig, variant_name
from finetuning.models import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER

In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")

In [ ]:
def find_tlp_checkpoints(config_glob="run_configs/tlp_*.yaml"):
    """Locate the checkpoint each tlp_* pretraining run left behind.
    """
    checkpoints = []
    for config_path in sorted(glob.glob(config_glob)):
        label = Path(config_path).stem
        output_dir = Path(load_run_config(config_path).output_dir)
        epochs = sorted(
            output_dir.glob("epoch-*.pt"),
            key=lambda p: int(p.stem.split("-")[1]),
        )
        if not epochs:
            print(f"  SKIP {label}: no epoch-*.pt under {output_dir}")
            continue
        print(f"  {label}: {epochs[-1]}")
        checkpoints.append((str(epochs[-1]), label))
    return checkpoints


print("tlp checkpoints:")
TLP_CHECKPOINTS = find_tlp_checkpoints()
print(f"\nfound {len(TLP_CHECKPOINTS)} of 4")

In [ ]:
BASELINES = [
    (BERT, "bert"),
    (BART, "bart"),
    (ROBERTA, "roberta"),
    (POLITICS, "politics"),
    (ideology_pt, "ideology"),
]

SEMEVAL_ROOT = "results_semeval"


def run_all_models(models, semeval_config, seed, root=SEMEVAL_ROOT):
    """Fine-tune each of `models` on one SemEval variant, under one seed.
    """
    # One directory per variant, so `agg.discover_results` reads a single comparison.
    loc = f"{root}/{variant_name(semeval_config)}/seed_{seed}"
    os.makedirs(loc, exist_ok=True)
    results = {}
    for model_ref, model_name in models:
        # 2814 train rows at the shared effective batch of 128 is 22 optimizer steps an
        # epoch, so this study needs more of them than MITweet did.
        exp = ExperimentConfig(patience=4, num_epochs=20, save_model=False, seed=seed)
        print(f"\n{'='*60}")
        print(f"Model: {model_name}  |  Variant: {variant_name(semeval_config)}  |  Seed: {seed}")
        print('='*60)
        results[model_name] = run_semeval_experiment(
            model=model_ref,
            loc=loc,
            semeval_config=semeval_config,
            experiment_config=exp,
            model_name=model_name,
        )
    return results

In [ ]:
seeds = [42, 1, 13, 1234, 6789]
MODELS = BASELINES + TLP_CHECKPOINTS

print(f"{len(MODELS)} models x {len(seeds)} seeds x {len(PREPEND_MODES)} prepend modes")
print("targets:", ALL_TARGETS)

## Stance detection, taskA

Every test target appears in training. `prepend="none"` is the control: without the target
the model is not told which of five questions it is being asked, so anything the `target`
arm gains over it is the prefix doing work.

In [ ]:
for prepend in PREPEND_MODES:
    for seed in seeds:
        print(f"\n{'#'*60}\nstance / {prepend} / taskA, seed: {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            semeval_config=SemEvalConfig(task="stance", prepend=prepend, split="taskA"),
            seed=seed,
        )

## Stance detection, taskB

Donald Trump, who appears in no training row and no validation row. This is the cross-target
arm: whatever a model scores here it learned from the other five targets. SemEval ran it as a
weakly supervised task for the same reason.

In [ ]:
for prepend in PREPEND_MODES:
    for seed in seeds:
        print(f"\n{'#'*60}\nstance / {prepend} / taskB, seed: {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            semeval_config=SemEvalConfig(task="stance", prepend=prepend, split="taskB"),
            seed=seed,
        )

## Opinion-target classification, taskA

The same tweets and the same splits, labelled by *who* the opinion is about rather than which
way it points. Selected on the plain 3-class `f1_macro`: `f1_favor_against` is a stance metric
and is not reported here.

In [ ]:
for prepend in PREPEND_MODES:
    for seed in seeds:
        print(f"\n{'#'*60}\nopinion / {prepend} / taskA, seed: {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            semeval_config=SemEvalConfig(task="opinion", prepend=prepend, split="taskA"),
            seed=seed,
        )

## Opinion-target classification, taskB

In [ ]:
for prepend in PREPEND_MODES:
    for seed in seeds:
        print(f"\n{'#'*60}\nopinion / {prepend} / taskB, seed: {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            semeval_config=SemEvalConfig(task="opinion", prepend=prepend, split="taskB"),
            seed=seed,
        )

## Cross-seed analysis

Everything below reads the per-seed metrics JSONs off disk — no model, no GPU. The runs above
do not need to have happened in this session.

Where the numbers live in these JSONs:

| | key | what it is |
|---|---|---|
| SemEval's own metric | `f1_favor_against` | mean of the FAVOR and AGAINST F1s, pooled over the split. **NONE is excluded by the shared task's definition**, not by oversight |
| 3-class macro | `f1_macro` | all three classes; the only headline number for the opinion task |
| per-target mean | `target_f1_macro` | the same score computed within each target, then averaged |
| per target | `f1_target_{i}` | `i` indexes `ALL_TARGETS` |

In [ ]:
VARIANT = "stance_target_taskA"

results = agg.discover_results(f"{SEMEVAL_ROOT}/{VARIANT}")
print(f"{len(results)} runs under {SEMEVAL_ROOT}/{VARIANT}")

# A model missing seeds gets a mean over fewer runs, and n is the only place that shows up.
agg.coverage(results)

In [ ]:
STANCE_METRICS = ["f1_favor_against", "f1_macro", "accuracy", "target_f1_macro", "target_accuracy"]
OPINION_METRICS = ["f1_macro", "accuracy", "target_f1_macro", "target_accuracy"]

agg.summary_table(results, metrics=STANCE_METRICS)

In [ ]:
# Every variant side by side: does naming the target help, and does that survive moving to a
# target the model never trained on? One row per (variant, model).
import pandas as pd

rows = []
for variant in sorted(os.listdir(SEMEVAL_ROOT)):
    runs = agg.discover_results(f"{SEMEVAL_ROOT}/{variant}")
    if not runs:
        continue
    metrics = STANCE_METRICS if variant.startswith("stance") else OPINION_METRICS
    table = agg.summary_table(runs, metrics=metrics)
    for model, row in table.iterrows():
        rows.append({"variant": variant, "model": model, **row.to_dict()})

pd.DataFrame(rows).set_index(["variant", "model"])

In [ ]:
# Per-target F1 for one model, so a variant that only helps the frequent targets is visible as
# such. On a stance variant these are SemEval's FAVOR/AGAINST average within each target.
import json

MODEL = "tlp_16"

runs = agg.by_model(agg.discover_results(f"{SEMEVAL_ROOT}/{VARIANT}"))[MODEL]
rows = []
for run in runs:
    payload = json.load(open(run.path))
    for target, name in enumerate(ALL_TARGETS):
        key = f"f1_target_{target}"
        if key in payload:
            rows.append({"target": name, "seed": run.seed, "f1": payload[key]})

frame = pd.DataFrame(rows)
frame.groupby("target")["f1"].agg(["mean", "std", "count"]).round(2)

In [ ]:
# The ensemble across seeds, how much the seeds disagree, and which class confusions dominate.
# The adjacent/polarity-flip labels read correctly for stance, where AGAINST(0) and FAVOR(2)
# sit either side of NONE(1); for the opinion task the three classes are not ordered, so read
# the pair counts and ignore the names.
agg.print_reports(agg.discover_results(f"{SEMEVAL_ROOT}/{VARIANT}"))